# 20.20 — Edge & on-device ML

Edge ML moves prediction onto the device, trading cloud control for privacy, offline availability, latency, memory, and battery constraints.

Compression supplies smaller artifacts, and cost-latency optimization supplies budget thinking. On-device design forces every MLOps decision into a small, user-owned environment. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 17
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

## The concept, built once

On-device memory and energy: $M=M_{model}+M_{runtime}+M_{features}$ and $E=e\,n$.

The reusable budget method computes quantized artifact size, total memory, daily energy, and SLA pass flags for a fleet of devices.

In [ ]:
def quantized_model_mb(parameter_count, bits):
    bytes_per_parameter = bits / 8
    return parameter_count * bytes_per_parameter / (1024 ** 2)

def edge_budget(model_mb, runtime_mb, feature_mb, energy_mj, n, memory_budget_mb=64, energy_budget_mj=8000):
    model = np.asarray(model_mb, dtype=float)
    runtime = np.asarray(runtime_mb, dtype=float)
    features = np.asarray(feature_mb, dtype=float)
    energy = np.asarray(energy_mj, dtype=float)
    calls = np.asarray(n, dtype=float)
    total_memory = model + runtime + features
    daily_energy = energy * calls
    memory_pass = total_memory <= memory_budget_mb
    energy_pass = daily_energy <= energy_budget_mj
    sla_pass = memory_pass & energy_pass
    return {
        "total_memory": total_memory,
        "daily_energy": daily_energy,
        "memory_pass": memory_pass,
        "energy_pass": energy_pass,
        "sla_pass": sla_pass,
        "pass_rate": float(np.mean(sla_pass)),
    }

lesson_memory = 18 + 12 + 6
lesson_energy = 120 * 50
offline_coverage = 0.15
download_seconds = 18 / 6
record_reduction = 1000 - 20
assert lesson_memory == 36
assert lesson_memory < 64
assert lesson_energy == 6000
assert abs(offline_coverage - 0.15) < 1e-12
assert download_seconds == 3
assert record_reduction == 980
print("memory MB", lesson_memory)
print("daily energy mJ", lesson_energy)
print("download seconds", download_seconds)

The same method now becomes the single evaluation API. It returns the operational artifact and the topic metric so the D1 proof scales to D2–D5.

In [ ]:
print("Build-once assertions passed for 20.20")

## The dataset ladder

Family F17 uses an inline D1–D5 systems workload ladder. Each rung increases operational realism while keeping the computation CPU-only, seeded, and NumPy based.

In [ ]:
def make_devices(n, rng, model_center, battery_stress=0.0, network_spikes=False):
    params = rng.normal(model_center, model_center * 0.12, size=n).clip(model_center * 0.5, None)
    bits = rng.choice([8, 6, 4], size=n, p=[0.55, 0.25, 0.20])
    model_mb = quantized_model_mb(params, bits)
    runtime_mb = rng.normal(12, 3, size=n).clip(6, None)
    feature_mb = rng.normal(6, 2, size=n).clip(1, None)
    energy = rng.normal(120, 25, size=n).clip(30, None)
    calls = rng.poisson(50, size=n) + 1
    if battery_stress > 0:
        energy = energy * (1 + battery_stress * rng.random(n))
    if network_spikes:
        feature_mb = feature_mb + rng.gamma(2, 2, size=n)
    device_class = rng.choice(["low", "mid", "high"], size=n, p=[0.35, 0.45, 0.20])
    memory_budget = np.where(device_class == "low", 48, np.where(device_class == "mid", 64, 96))
    return {"model_mb": model_mb, "runtime_mb": runtime_mb, "feature_mb": feature_mb, "energy": energy, "calls": calls, "class": device_class, "memory_budget": memory_budget, "bits": bits}

def build_edge_ladder(seed=17):
    rng = np.random.default_rng(seed)
    ladder = []
    ladder.append({"name": "D1 toy device budget", "devices": {"model_mb": np.array([18.0]), "runtime_mb": np.array([12.0]), "feature_mb": np.array([6.0]), "energy": np.array([120.0]), "calls": np.array([50]), "class": np.array(["mid"]), "memory_budget": np.array([64.0]), "bits": np.array([8])}, "load": 1})
    ladder.append({"name": "D2 clean 1k device requests", "devices": make_devices(1000, rng, 18000000), "load": 1000})
    ladder.append({"name": "D3 battery and network spikes", "devices": make_devices(2500, rng, 22000000, 0.40, True), "load": 2500})
    ladder.append({"name": "D4 sampled fleet classes", "devices": make_devices(7000, rng, 26000000, 0.20, True), "load": 7000})
    ladder.append({"name": "D5 production rollout simulation", "devices": make_devices(15000, rng, 32000000, 0.55, True), "load": 15000})
    return ladder

ladder = build_edge_ladder()
for rung in ladder:
    devices = rung["devices"]
    print(rung["name"], "n", len(devices["model_mb"]), "model sample MB", np.round(devices["model_mb"][:5], 2).tolist())

## Run the same method across D1–D5

The metric is collected in the same way for every rung, so changes across the ladder reflect workload complexity rather than a different measurement.

In [ ]:
results = []
for rung in ladder:
    devices = rung["devices"]
    budgets = devices["memory_budget"]
    result = edge_budget(devices["model_mb"], devices["runtime_mb"], devices["feature_mb"], devices["energy"], devices["calls"], memory_budget_mb=budgets)
    p95_size = float(np.quantile(result["total_memory"], 0.95))
    results.append({"name": rung["name"], "metric": p95_size, "artifact": result, "load": rung["load"], "devices": devices})

print("rung | p95 on-device size MB | SLA pass rate | median energy")
for item in results:
    artifact = item["artifact"]
    print(item["name"], round(item["metric"], 2), round(artifact["pass_rate"], 3), round(float(np.median(artifact["daily_energy"])), 1))

## Results visualization

The closing figure has operational panels for each rung plus a metric-vs-load curve.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for col, item in enumerate(results):
    artifact = item["artifact"]
    devices = item["devices"]
    axes[0, col].hist(artifact["total_memory"], bins=30, color="tab:blue")
    axes[0, col].set_title(item["name"].split()[0] + " memory")
    axes[0, col].set_xlabel("MB")
    axes[0, col].set_ylabel("devices")
    pass_by_bits = []
    labels = []
    for bits in sorted(np.unique(devices["bits"])):
        mask = devices["bits"] == bits
        labels.append(str(bits) + " bit")
        pass_by_bits.append(float(np.mean(artifact["sla_pass"][mask])))
    axes[1, col].bar(labels, pass_by_bits, color="tab:pink")
    axes[1, col].set_ylim(0, 1)
    axes[1, col].set_ylabel("SLA pass")
fig.suptitle("Operational panels: on-device size and quantized SLA pass")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(7, 4))
loads = [item["load"] for item in results]
metrics = [item["metric"] for item in results]
ax.plot(loads, metrics, marker="o")
ax.set_xlabel("fleet size")
ax.set_ylabel("p95 on-device size MB")
ax.set_title("Metric vs fleet size")
plt.show()

## Pitfall on the hardest rung

Reproduce the lesson pitfall on D5, then apply the operational fix and compare the metric.

In [ ]:
d5 = results[-1]
devices = d5["devices"]
model_only_pass = devices["model_mb"] <= devices["memory_budget"]
total_pass = d5["artifact"]["memory_pass"]
print("wrong model-only memory pass", round(float(np.mean(model_only_pass)), 3))
print("correct total-memory pass", round(float(np.mean(total_pass)), 3))
print("D5 p95 on-device size MB", round(float(d5["metric"]), 2))
print("SLA pass after energy too", round(float(d5["artifact"]["pass_rate"]), 3))

## Evaluate it + Practice

- Metric: p95 on-device size in MB, compared with model-size-only accounting.
- Sanity check: 18+12+6 gives 36 MB and 120 mJ times 50 gives 6,000 mJ/day.
- Ablation: switch from 4-bit or 6-bit artifacts back to 8-bit and watch device pass rate fall.
- Failure signals: memory-budget misses, update-download size, and energy spikes on low-end devices.

Practice prompts:
1. Change one workload knob on D3 and predict whether the metric should improve or degrade.
2. Add one slice or route to D4 and explain which operational panel should change.
3. Tighten the D5 budget or threshold and rerun only after saving your own copy.